In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd

from frigeo import racine_projet
from frigeo.perimetre import decouvrir

# PARAMETRE A MODIFIER CHAQUE MOIS
# Dernier mois déposé dans data/raw, au format AAAA-MM
PERIODE_FIN = "2026-03"

In [ ]:
DATA = racine_projet() / "data"
perimetre = decouvrir(DATA, PERIODE_FIN)
perimetre.verifier()
print(perimetre.resume())
perimetre.tableau()

In [ ]:
from frigeo.chargement import charger_tout
from frigeo.profilage import profiler_tout, detail_signatures

tables = charger_tout(perimetre)
profil = profiler_tout(tables)
print(profil.shape)

In [ ]:
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 250)
display(profil[profil["table"] == "interventions"].drop(columns=["table"]))

In [ ]:
for colonne in ["date_debut", "duree_min", "code_client", "lieu_cp"]:
    print(colonne)
    display(detail_signatures(tables["interventions"], colonne, n=10))

In [ ]:
interventions = tables["interventions"]
for colonne in ["categorie", "statut", "facturable", "lieu_ville"]:
    print(colonne)
    display(interventions[colonne].map(repr).value_counts().rename("effectif").to_frame())

In [ ]:
ref = tables["intervenants"]
attendus = set(ref["nom"].astype(str) + " " + ref["prenom"].astype(str))
tableau = interventions["intervenant"].value_counts().rename("effectif").to_frame()
tableau["dans_referentiel"] = tableau.index.isin(attendus)
tableau.index = tableau.index.map(repr)
display(tableau)

In [ ]:
display(profil[profil["table"] == "pieces_interventions"].drop(columns=["table"]))

In [ ]:
display(profil[profil["table"] == "fec"].drop(columns=["table"]))

In [ ]:
for colonne in ["Debit", "PieceRef"]:
    print(colonne)
    display(detail_signatures(tables["fec"], colonne, n=15))

In [ ]:
display(profil[profil["table"] == "clients"].drop(columns=["table"]))

In [ ]:
petits = ["fournisseurs", "intervenants", "remunerations", "catalogue_pieces", "parametres_couts"]
display(profil[profil["table"].isin(petits)])

In [ ]:
factures = ["factures_entetes", "factures_lignes"]
display(profil[profil["table"].isin(factures)])

In [ ]:
for table, colonne in [("factures_entetes", "date_facture"), ("factures_entetes", "total_ht"), ("factures_lignes", "pu_ht")]:
    print(table, colonne)
    display(detail_signatures(tables[table], colonne, n=15))

In [ ]:
colonnes = [
    ("factures_entetes", "total_ht"),
    ("factures_entetes", "total_ttc"),
    ("factures_lignes", "pu_ht"),
    ("factures_lignes", "montant_ht"),
]
for table, colonne in colonnes:
    detail = detail_signatures(tables[table], colonne, n=30)
    print(table, colonne)
    display(detail[detail["type"] == "str"])

In [ ]:
achats = ["achats_entetes", "achats_lignes"]
display(profil[profil["table"].isin(achats)])

### Synthèse du profilage

Le profil porte sur 13 tables et 131 colonnes de données du périmètre courant. Il décrit ce qui est présent, sans rien corriger. Les familles d'écarts constatées, à confirmer et à localiser par les contrôles de l'étape 3, sont les suivantes.

- Champs obligatoires vides (chaînes vides dans les CSV et le FEC, valeurs absentes dans Excel).
- Dates dans des formats autres que `JJ/MM/AAAA`, dates en texte dans Excel et dates très postérieures à la période.
- Montants en texte dans Excel et montants du FEC hors format (point décimal, séparateur de milliers).
- Valeurs hors listes fermées (casse, accents, espaces, abréviations, synonymes) et noms d'intervenants non conformes au référentiel.
- Codes et identifiants mal formés (clients, codes postaux, SIRET).
- Encodages mélangés dans les interventions, le FEC et les clients.
- Signaux de doublons dans les identifiants, les référentiels et les références de factures fournisseurs.